**Ejercicio** **1**

Consignas:

Lea en Colab el archivo flete-aereo-vacunas-covid19-al-2021-06-28.xlsx y realice cualquier tarea de limpieza y/o adecuación del dataset que considere necesaria.

1- Según la información que se encuentra en la base de datos, ¿cuál fue el número de vuelo que realizó una mayor cantidad de fletes? Sugerencia: utilice el método value_counts() de la librería Pandas.

2- ¿Cuántos registros no contienen información sobre el número de vuelo?

3- Calcule el promedio de lo facturado en todos los vuelos realizados considerando únicamente los registros de viajes que se facturaron en USD según la información contenida en la columna factura_moneda_monto.

4- ¿Qué porcentaje de los vuelos realizados tuvieron como origen a Rusia o a China?

5- ¿Cuál es el vuelo más reciente de los que se tiene registro en el dataset? ¿Cuántos días transcurrieron entre el primer y el último vuelo realizados?

6- Escribir el archivo en formato .parquet.

# Lectura limpieza y adecuación:

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from google.colab import drive

drive.mount('/content/gdrive/', force_remount=True)


In [3]:
df = pd.read_excel('/content/drive/MyDrive/Datasets_fundamentos/flete-aereo-vacunas-covid19-al-2021-06-28.xlsx')


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 66 entries, 0 to 65
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   organismo                       66 non-null     object        
 1   expediente                      66 non-null     object        
 2   financiamiento                  66 non-null     object        
 3   acto_conclusión                 66 non-null     object        
 4   descripción                     66 non-null     object        
 5   prestador                       66 non-null     object        
 6   CUIT                            66 non-null     object        
 7   estado                          66 non-null     object        
 8   factura_nro                     66 non-null     object        
 9   factura_moneda_monto            64 non-null     object        
 10  guía_nro                        66 non-null     object        
 11  guía_moneda_m

In [7]:

'''
acá vemos que las primeras filas están con NAN, vamos a usar la funcion header para abrir el dataset a partir
de la 4ta fila.
'''

df.head()

df = pd.read_excel('/content/drive/MyDrive/Datasets_fundamentos/flete-aereo-vacunas-covid19-al-2021-06-28.xlsx', header=4)
df.head()


,organismo,expediente,financiamiento,acto_conclusión,descripción,prestador,CUIT,estado,factura_nro,factura_moneda_monto,guía_nro,guía_moneda_monto,fecha_guía,vuelo,bienes_transportados,proveedor_bienes_transportados,operación,comprador_donante,origen
0,Ministerio de Salud de la Nación,EX-2020-91578184- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-16691129-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011116,"USD 305.250,00",044-44032951,"USD 679,84",2020-12-23,1061,marking,Biointegrator LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
1,Ministerio de Salud de la Nación,EX-2020-91578184- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-16691129-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011116,"USD 302.250,00",044-44032940,"USD 304.570,16",2020-12-23,1061,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
2,Ministerio de Salud de la Nación,EX-2021-04601170- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-24666588-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011206,"USD 302.250,00",044-44032973,"USD 302.250,00",2021-01-15,1061,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
3,Ministerio de Salud de la Nación,EX-2021-08460702- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-24639513-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011296,NaN,044-44032984,"USD 268.655,94",2021-01-29,1063,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
4,Ministerio de Salud de la Nación,EX-2021-13211127- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-24642675-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011370,"USD 303.075,00",044-44032995,"USD 303.075,00",2021-02-11,1065,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia


In [8]:
'''
vamos a ver los nombres de la columnas... si miramos en detalle vemos que aparece un espacio al final de vuelos.
hay que corregir esto.
'''
df.columns

Index(['organismo', 'expediente', 'financiamiento', 'acto_conclusión',
       'descripción', 'prestador', 'CUIT', 'estado', 'factura_nro',
       'factura_moneda_monto', 'guía_nro', 'guía_moneda_monto', 'fecha_guía',
       'vuelo ', 'bienes_transportados', 'proveedor_bienes_transportados',
       'operación', 'comprador_donante', 'origen'],
      dtype='object')

In [9]:
'''
una forma rápida de cambiar esto es usar el metodo rename. Pero acá se encadena otro método que arma pares
de datos ordenados (zip) entre los datos originales y todos los nombres de las columnas sin espacios.
'''
df.rename(columns={old_name: new_name for old_name,
                   new_name in zip(df.columns, [col.replace(' ', '') for col in df.columns])},
                   inplace=True)


In [11]:
df.columns

#acá ya se ve corregido

Index(['organismo', 'expediente', 'financiamiento', 'acto_conclusión',
       'descripción', 'prestador', 'CUIT', 'estado', 'factura_nro',
       'factura_moneda_monto', 'guía_nro', 'guía_moneda_monto', 'fecha_guía',
       'vuelo', 'bienes_transportados', 'proveedor_bienes_transportados',
       'operación', 'comprador_donante', 'origen'],
      dtype='object')

In [16]:
'''
punto 1 - Según la información que se encuentra en la base de datos,
 ¿cuál fue el número de vuelo que realizó una mayor cantidad de fletes?
'''

# Calcula la frecuencia de cada vuelo
frecuencia_vuelos = df['vuelo'].value_counts()

#print(frecuencia_vuelos)

#Calcula el porcentaje de cada vuelo
porcentaje_vuelos = (frecuencia_vuelos / df['vuelo'].count())

# Muestra los resultados
print(porcentaje_vuelos)

#El vuelo 1061 fue el que realizó mayor cantidad de fletes.


vuelo
1061                   0.238095
1063                   0.190476
1065                   0.142857
1051                   0.111111
1081                   0.095238
1067                   0.063492
1069                   0.047619
1501                   0.031746
QR8351/16-QR8155/16    0.015873
KL701                  0.015873
KL892                  0.015873
LH8433                 0.015873
LH8434                 0.015873
Name: count, dtype: float64


In [25]:
'''
punto 2 cuantos registros no contienen información sobre el número de vuelo?
'''

#vamos a ver qué tipo de datos tenemos en la columna vuelo
print(df.vuelo.unique())

# Contar los registros donde 'vuelo' es NaN (no contiene información)
registros_sin_vuelo = df['vuelo'].isna().sum()

print(f"Número de registros sin información sobre el número de vuelo: {registros_sin_vuelo}")

#este número es totalmente exagerado. Veamos bien el data frame.

[1061 1063 1065 'QR8351/16-QR8155/16' 1051 nan 1501 'KL892' 1067 'KL701'
 1069 1081 'LH8433' 'LH8434']
Número de registros sin información sobre el número de vuelo: 3


In [24]:
df

# las ultimas filas son casi todos los valores NaN. Hay que limpiar esto.

df.dropna(axis=0,  thresh=3, inplace=True)
df
# ahora el dataset tiene 66 filas y ésto resulta razonable.

,organismo,expediente,financiamiento,acto_conclusión,descripción,prestador,CUIT,estado,factura_nro,factura_moneda_monto,guía_nro,guía_moneda_monto,fecha_guía,vuelo,bienes_transportados,proveedor_bienes_transportados,operación,comprador_donante,origen
0,Ministerio de Salud de la Nación,EX-2020-91578184- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-16691129-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011116,"USD 305.250,00",044-44032951,"USD 679,84",2020-12-23,1061,marking,Biointegrator LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
1,Ministerio de Salud de la Nación,EX-2020-91578184- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-16691129-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011116,"USD 302.250,00",044-44032940,"USD 304.570,16",2020-12-23,1061,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
2,Ministerio de Salud de la Nación,EX-2021-04601170- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-24666588-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011206,"USD 302.250,00",044-44032973,"USD 302.250,00",2021-01-15,1061,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
3,Ministerio de Salud de la Nación,EX-2021-08460702- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-24639513-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011296,NaN,044-44032984,"USD 268.655,94",2021-01-29,1063,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
4,Ministerio de Salud de la Nación,EX-2021-13211127- -APN-DD#MS,Fuente Financiamiento 11,DI-2021-24642675-APN-SSGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011370,"USD 303.075,00",044-44032995,"USD 303.075,00",2021-02-11,1065,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,Ministerio de Salud de la Nación,EX-2021-51336146- -APN-DD#MS,Fuente Financiamiento 13,RS-2021-54995747-APN-SGA#MS,Logística,DHL GLOBAL FORWARDING (ARGENTINA) S.A.,30-58471593-2,Cumplido,0028-00000784,"EUR 331.069,72",044-44033220,"EUR 331.069,72",2021-06-03,NaN,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
62,Ministerio de Salud de la Nación,EX-2021-51336146- -APN-DD#MS,Fuente Financiamiento 13,RS-2021-54995747-APN-SGA#MS,Seguro,DHL GLOBAL FORWARDING (ARGENTINA) S.A.,30-58471593-2,Cumplido,0028-00000785,"USD 19.700,23",044-44033220,"USD 19.700,23",2021-06-03,1061,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
63,Ministerio de Salud de la Nación,EX-2021-52119597- -APN-DD#MS,Fuente Financiamiento 13,RS-2021-56095206-APN-SGA#MS,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011853,"USD 259.052,99",044-44033231,"USD 259.052,99",2021-06-08,1063,vaccine pharmaceuticals,Human Vaccine LLC,Adquisición,Secretaría de Acceso a la Salud,Rusia
64,Ministerio de Salud de la Nación,EX-2021-56095077- -APN-DD#MS,Fuente Financiamiento 13,sin acto de conclusión,Flete aéreo,Aerolíneas Argentinas S.A.,30-64140555-4,Cumplido,0411-00011897,USD 428.000,044-44033264,"USD 14.687,34",2021-06-21,1065,sars-cov-2 vaccine (vero cell),Beijin Institute of Biological Phroducts co.,Adquisición,Secretaría de Acceso a la Salud,China


In [27]:
'''
punto 3, Calcule el promedio de lo facturado en todos los  los vuelos
realizados considerando únicamente los registros de viajes que se facturaron en USD
según la información contenida en la columna factura_moneda_monto.
'''

# Extrae las primeras tres letras y el número de la cadena
df['Moneda'] = df['factura_moneda_monto'].str.extract(r'([A-Z]+)')
df['Monto'] = df['factura_moneda_monto'].str.extract(r'(\d+\.?\d*,\d+)')

# Procesa el número para eliminar el punto de los miles y reemplazar la coma decimal con un punto
df['Monto'] = df['Monto'].str.replace('.', '').str.replace(',', '.')

# Convertir la columna 'Monto' de tipo 'O' a tipo numérico
df['Monto'] = pd.to_numeric(df['Monto'], errors='coerce')

# Filtrar las filas donde la moneda es USD
montos_usd = df.loc[df['Moneda'] == 'USD', 'Monto']

# Calcular el promedio de lo facturado en EUR y USD
promedio_usd = montos_usd.mean()

# Imprimir el promedio:
print("Promedio de lo facturado en USD:", round(promedio_usd,1))


Promedio de lo facturado en USD: 203181.2


In [36]:
'''
punto 4: ¿Qué porcentaje de los vuelos tuvo como origen a Rusia o a China?
'''

porcentaje_ruso_chino= len(df[(df['origen']=='Rusia')|(df['origen']=='China')])/len(df) * 100

print("El porcentaje de vuelos con origen Rusia/China es:", round(porcentaje_ruso_chino,2),"%")


El porcentaje de vuelos con origen Rusia/China es: 95.45 %


In [37]:
'''
5, ¿Cuál es el vuelo más reciente de los que se tiene registro en el dataset?
¿Cuántos días transcurrieron entre el primer y el último vuelo?
'''

print('El ultimo vuelo fue: ',max(df['fecha_guía']))
print('El primer vuelo fue: ',min(df['fecha_guía']))
print('Cantidad de dias entre vuelos: ', max(df['fecha_guía'])- min(df['fecha_guía']))

El ultimo vuelo fue:  2021-06-22 00:00:00
El primer vuelo fue:  2020-12-23 00:00:00
Cantidad de dias entre vuelos:  181 days 00:00:00


In [40]:
'''
6, convertir el data frame a formato parquet.
'''

# Convertir la columna 'vuelo' a tipo de datos cadena (str)
df['vuelo'] = df['vuelo'].astype(str)

# Escribir el DataFrame en formato Parquet
df.to_parquet('vuelos_covid.parquet')